# Metric Learning Experiments

Train and evaluate learned embedding models on governed datasets, comparing
against the `texture_baseline_v0.2.0` hand-crafted feature baseline.

> **Warning:** Results are diagnostic only. Public datasets (FabricNET, LeaData)
> are not luxury authentication datasets. Learned model performance here does
> not prove luxury authentication accuracy.

## Curated Results

See [learned_metric_v0.2.0 Results](../docs/datasets/learned-metric-v02-results.md)
for the full curated summary including comparison against texture baselines and
recommended next experiments.

## Run Experiments

From `auth-platform/`:

```bash
# LeaData (16 species classes, primary experiment)
make learned-metric-experiment-leadata-v02

# Quick LeaData (2 epochs, 100 images)
make learned-metric-experiment-leadata-quick

# FabricNET (limited — 1 image per class, see config notes)
make learned-metric-experiment-fabricnet-v02

# Compare learned vs baseline
make compare-learned-vs-baseline DATASET=external_leadata_v0.1.0
```

In [ ]:
import json
from pathlib import Path

%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt

# Configuration
DATASET = "external_leadata_v0.1.0"
LEARNED_DIR = Path("../experiments/outputs/learned-metric")
BASELINE_DIR = Path("../experiments/outputs/baseline-feature-comparison")

ds_tag = DATASET.replace(".", "_")

# Find learned metric results
learned_files = sorted(LEARNED_DIR.glob(f"{ds_tag}*__experiment_record.json")) if LEARNED_DIR.exists() else []
training_files = sorted(LEARNED_DIR.glob(f"{ds_tag}*__training_metrics.json")) if LEARNED_DIR.exists() else []

print(f"Learned records: {len(learned_files)}")
print(f"Training files:  {len(training_files)}")

In [ ]:
# Load experiment record
if learned_files:
    rec = json.loads(learned_files[-1].read_text())
    print(f"Model:      {rec.get('model_version')}")
    print(f"Dataset:    {rec.get('dataset_version')}")
    print(f"Backbone:   {rec.get('backbone')} (pretrained={rec.get('pretrained')})")
    print(f"Emb dim:    {rec.get('embedding_dim')}")
    print(f"Epochs:     {rec.get('epochs')}")
    print(f"Classes:    {rec.get('n_classes')}")
    print(f"Train time: {rec.get('training_time_seconds')}s")
    em = rec.get("evaluation_metrics", {})
    if em:
        print(f"\nTest evaluation:")
        print(f"  Cluster sep: {em.get('cluster_separation')}")
        print(f"  NN agree:    {em.get('nn_agreement_rate')}")
        print(f"  Recall@1:    {em.get('recall_at_1')}")
else:
    rec = None
    print("No learned metric results found. Run experiment first.")

In [ ]:
# Training loss curve
if training_files:
    tm = json.loads(training_files[-1].read_text())
    train_losses = tm.get("train_losses", [])
    val_losses = tm.get("val_losses", [])

    if train_losses:
        fig, ax = plt.subplots(figsize=(10, 5))
        ax.plot(range(1, len(train_losses) + 1), train_losses, label="Train loss", color="steelblue")
        if val_losses:
            ax.plot(range(1, len(val_losses) + 1), val_losses, label="Val loss", color="tomato")
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Loss")
        ax.set_title(f"Training Loss — {DATASET}")
        ax.legend()
        plt.tight_layout()
        plt.show()
else:
    print("No training metrics to plot.")

In [ ]:
# Comparison: learned vs baseline
baseline_files = sorted(BASELINE_DIR.glob(f"{ds_tag}*__record.json")) if BASELINE_DIR.exists() else []

if rec and rec.get("evaluation_metrics") and baseline_files:
    em = rec["evaluation_metrics"]
    print(f"{'Method':<35} {'Dim':>5} {'ClustSep':>10} {'NN Agree':>10} {'R@1':>8}")
    print("-" * 75)

    for bf in baseline_files:
        br = json.loads(bf.read_text())
        bm = br.get("metrics", {})
        print(f"{br.get('feature_extractor', '?'):<35} {bm.get('dimension', ''):>5} "
              f"{bm.get('cluster_separation', 0):>10.4f} "
              f"{bm.get('nn_agreement_rate', 0):>9.1%} "
              f"{bm.get('recall_at_1', 0):>8.4f}")

    name = f"{rec.get('model_version', 'learned')} ({rec.get('backbone', '')})"
    print(f"{name:<35} {em.get('dimension', ''):>5} "
          f"{em.get('cluster_separation', 0):>10.4f} "
          f"{em.get('nn_agreement_rate', 0):>9.1%} "
          f"{em.get('recall_at_1', 0):>8.4f}")
else:
    print("Need both learned and baseline results for comparison.")

## Notes / Conclusions

- `learned_metric_v0.2.0` uses pretrained ResNet18 with contrastive loss.
- LeaData (16 species) provides meaningful multi-class structure for contrastive training.
- FabricNET (1 image per sample) has limited training value for supervised approaches.
- Compare cluster separation and NN agreement against `texture_baseline_v0.2.0`.
- Backbone comparison (ResNet50, ConvNeXt, etc.) is future work — see notebook 05.

**These results are diagnostic only.** They do not prove authentication accuracy.